# 01 · E1 Component comparison (ULB)

Trains IF, OCSVM and the autoencoder on each of the five splits and scores six configurations: the three detectors on their own, RAREGuard without feature selection, without PCA, and the full pipeline. Every configuration gets two thresholds:

* **Fully unsupervised**: the 99.8th percentile of the validation scores (c = 0.002). No validation labels.
* **Label-assisted**: the validation-F1-best percentile. Only there as a reference point.

The single detectors are taken from the same fit as the full pipeline (IF_Only is the IF inside RAREGuard_Full), so the comparison is matched. IF and OCSVM are deterministic given the seed, so this gives the same numbers as fitting them separately.

This notebook also saves the normalised component scores of the full pipeline for every seed (`scores/ulb_seed*.npz`). Notebooks 02, 03 and 05 reuse them, so fusion rules and thresholds are compared on identical scores without retraining.

| | |
|---|---|
| Trains models | yes: 3 feature pipelines × 3 detectors × 5 seeds |
| Expected time | roughly 20 min on a GPU runtime (estimated from the v5 training times) |
| Writes | `results/E1_components/`: `runs_by_seed.csv`, `summary_mean_sd.csv`, `features_by_seed.csv`, `scores/ulb_seed*.npz` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import gc, time
import numpy as np, pandas as pd

EXP = "E1_components"
FORCE_RERUN = False    # True = train again; the previous files are moved to _archive/

RUN_TRAINING = FORCE_RERUN or not rg.already_done(cfg, EXP, ["runs_by_seed.csv"])
print("Training will run." if RUN_TRAINING else "E1 results exist - loading them instead of retraining.")

In [ ]:
X, y, feature_names = rg.load_ulb(cfg)
print(f"ULB: {len(y):,} transactions, {int(y.sum())} frauds")

PIPELINES = {
    "full":   dict(use_feature_selection=True,  use_pca=True),
    "no_fs":  dict(use_feature_selection=False, use_pca=True),
    "no_pca": dict(use_feature_selection=True,  use_pca=False),
}
# configuration: (feature pipeline, detectors it uses)
CONFIGS = {
    "IF_Only":          ("full",   ["IF"]),
    "OCSVM_Only":       ("full",   ["OCSVM"]),
    "AE_Only":          ("full",   ["AE"]),
    "RAREGuard_No_FS":  ("no_fs",  ["IF", "OCSVM", "AE"]),
    "RAREGuard_No_PCA": ("no_pca", ["IF", "OCSVM", "AE"]),
    "RAREGuard_Full":   ("full",   ["IF", "OCSVM", "AE"]),
}
W = cfg["fusion_weights"]
C = cfg["contamination"]

## Train and evaluate

In [ ]:
if RUN_TRAINING:
    run = rg.Run(cfg, EXP, "01_E1_components.ipynb", "Component comparison on ULB, both threshold modes")
    rows, feature_rows = [], []

    for seed in cfg["seeds"]:
        tr, va, te = rg.get_split(cfg, "ulb", y, seed)
        y_tr, y_va, y_te = y[tr], y[va], y[te]

        fitted = {}
        for pipe, opts in PIPELINES.items():
            F = rg.prepare_features(X[tr], X[va], X[te], feature_names, seed, cfg, **opts)
            scores, secs, info = rg.fit_detectors(F, y_tr, seed, cfg)
            fitted[pipe] = (F, scores, secs)
            feature_rows.append({"Seed": seed, "Pipeline": pipe, "Selected_N": len(F["selected"]),
                                 "Selected_Features": "; ".join(F["selected"]), "Model_Dims": F["dims"],
                                 "Prep_s": F["seconds"], **{f"{m}_s": s for m, s in secs.items()},
                                 "AE_Epochs": info.get("AE_epochs")})
            print(f"seed {seed:>4} | {pipe:<6} | dims {F['dims']:>2} | prep {F['seconds']:5.1f}s | "
                  + "  ".join(f"{m} {s:5.1f}s" for m, s in secs.items()))

        for config, (pipe, members) in CONFIGS.items():
            F, scores, secs = fitted[pipe]
            if len(members) == 1:
                val_s, test_s = scores[members[0]][1], scores[members[0]][2]
            else:
                val_s = rg.weighted_fusion({m: scores[m][1] for m in members}, W)
                test_s = rg.weighted_fusion({m: scores[m][2] for m in members}, W)
            train_s = sum(secs[m] for m in members)

            for mode in ("Fully unsupervised", "Label-assisted"):
                if mode == "Fully unsupervised":
                    thr, val_f1 = rg.percentile_threshold(val_s, C), np.nan
                else:
                    thr, val_f1 = rg.label_assisted_threshold(y_va, val_s, cfg)
                rows.append({"Seed": seed, "Experiment": config, "Threshold_Mode": mode,
                             "Contamination": C if mode == "Fully unsupervised" else np.nan,
                             "Pipeline": pipe, "Model_Dims": F["dims"], "Validation_F1": val_f1,
                             "Train_Time_s": train_s, "Total_Time_s": F["seconds"] + train_s,
                             **rg.evaluate(y_te, test_s, thr, val_s)})

        # matched component scores of the full pipeline, reused by notebooks 02, 03, 05 and 09
        F, scores, _ = fitted["full"]
        run.save_scores(f"ulb_seed{seed}", y_val=y_va, y_test=y_te, val_idx=va, test_idx=te,
                        **{f"{m}_{part}": scores[m][i]
                           for m in scores for i, part in enumerate(("train", "val", "test"))})
        del fitted
        gc.collect()

    runs = pd.DataFrame(rows)
    run.save_csv(runs, "runs_by_seed.csv")
    run.save_csv(rg.mean_sd(runs, ["Threshold_Mode", "Experiment"],
                            rg.METRICS + ["Train_Time_s", "Total_Time_s"]), "summary_mean_sd.csv")
    run.save_csv(pd.DataFrame(feature_rows), "features_by_seed.csv")
    run.finish()

`Train_Time_s` counts only the detectors a configuration uses (IF_Only: the IF fit). `Total_Time_s` adds feature preparation. Splitting and evaluation are not timed.

## Check and quick look

In [ ]:
runs = rg.read_result(cfg, EXP, "runs_by_seed.csv")
assert len(runs) == len(cfg["seeds"]) * len(CONFIGS) * 2, "some seed/configuration rows are missing"
assert runs.groupby("Seed").Test_Fraud.nunique().eq(1).all()

summary = rg.read_result(cfg, EXP, "summary_mean_sd.csv")
cols = ["Experiment", "F1_mean", "F1_sd", "PR_AUC_mean", "PR_AUC_sd", "ROC_AUC_mean", "Train_Time_s_mean"]
for mode in ("Fully unsupervised", "Label-assisted"):
    print(mode)
    display(summary.loc[summary.Threshold_Mode == mode, cols].round(4))

In [ ]:
# Which features did the Laplacian score keep? Stable choices across seeds are reassuring.
feats = rg.read_result(cfg, EXP, "features_by_seed.csv")
full = feats[feats.Pipeline == "full"]
counts = pd.Series("; ".join(full.Selected_Features).split("; ")).value_counts()
print("PCA dimensions per seed:", dict(zip(full.Seed, full.Model_Dims)))
print("Features kept in all seeds:", sorted(counts[counts == len(full)].index))